In [ ]:
# setup
import torch
import torchvision.transforms as transforms
from PIL import Image
from pathlib import Path
from ultralytics import YOLO
import timm
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# config - model weights, dataset path, and constants shared across both models
YOLO_WEIGHTS         = '/content/drive/MyDrive/Colab Notebooks/fyp_notebooks/model_weights/yolov8.pt'
EFFICIENTNET_WEIGHTS = "/content/drive/MyDrive/Colab Notebooks/fyp_notebooks/model_weights/aggregated_efficientnet_b3.pth"
TEST_DIR             = "/content/drive/MyDrive/Data for Colab/aggregated_dataset/test"

EFFICIENTNET_VARIANT = "efficientnet_b3"
IMG_SIZE             = 224
CONF_THRESHOLD       = 0.25

CLASS_NAMES = [
    'Acne and Rosacea Photos',
    'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions',
    'Atopic Dermatitis Photos',
    'Bullous Disease Photos',
    'Cellulitis Impetigo and other Bacterial Infections',
    'Eczema Photos',
    'Exanthems and Drug Eruptions',
    'Hair Loss Photos Alopecia and other Hair Diseases',
    'Herpes HPV and other STDs Photos',
    'Light Diseases and Disorders of Pigmentation',
    'Lupus and other Connective Tissue diseases',
    'Melanoma Skin Cancer Nevi and Moles',
    'Nail Fungus and other Nail Disease',
    'Poison Ivy Photos and other Contact Dermatitis',
    'Psoriasis pictures Lichen Planus and related diseases',
    'Scabies Lyme Disease and other Infestations and Bites',
    'Seborrheic Keratoses and other Benign Tumors',
    'Squamous_Cell_Carcinoma',
    'Systemic Disease',
    'Tinea Ringworm Candidiasis and other Fungal Infections',
    'Urticaria Hives',
    'Vascular Tumors',
    'Vasculitis Photos',
    'Warts Molluscum and other Viral Infections'
]

In [ ]:
# load both models and define the preprocessing transform
from torchvision import models

yolo = YOLO(YOLO_WEIGHTS)

efficientnet = models.efficientnet_b3(weights=None)
efficientnet.classifier[1] = torch.nn.Linear(efficientnet.classifier[1].in_features, len(CLASS_NAMES))
efficientnet.load_state_dict(torch.load(EFFICIENTNET_WEIGHTS, map_location=device))
efficientnet = efficientnet.to(device)
efficientnet.eval()

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Models loaded")

In [ ]:
# classify runs EfficientNet on a (possibly cropped) image
# pipeline_predict runs YOLO first to isolate the lesion region, then classifies the crop
# falls back to the full image if YOLO detects nothing
def classify(pil_image):
    tensor = transform(pil_image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(efficientnet(tensor), dim=1)
    return CLASS_NAMES[probs.argmax(dim=1).item()]


def pipeline_predict(img_path):
    image = Image.open(img_path).convert("RGB")
    detections = yolo(str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]

    detected = False
    if len(detections.boxes) > 0:
        best_box = detections.boxes[detections.boxes.conf.argmax()]
        x1, y1, x2, y2 = map(int, best_box.xyxy[0].tolist())
        pad = 10
        crop = image.crop((
            max(0, x1 - pad), max(0, y1 - pad),
            min(image.width, x2 + pad), min(image.height, y2 + pad)
        ))
        detected = True
    else:
        crop = image

    return classify(crop), detected

In [ ]:
# evaluate pipeline vs baseline across the full test set
# baseline: EfficientNet on the full image (no YOLO)
# pipeline: YOLO crops the lesion region first, then EfficientNet classifies the crop
records = []
test_root = Path(TEST_DIR)

for class_dir in sorted(test_root.iterdir()):
    if not class_dir.is_dir():
        continue
    for img_path in tqdm(list(class_dir.glob("*")), desc=class_dir.name):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue
        image = Image.open(img_path).convert("RGB")
        base_pred = classify(image)
        pipe_pred, detected = pipeline_predict(img_path)
        records.append({
            "true":          class_dir.name,
            "base_pred":     base_pred,
            "pipe_pred":     pipe_pred,
            "yolo_detected": detected
        })

df = pd.DataFrame(records)
print(f"Done - {len(df)} images evaluated")

In [ ]:
# accuracy comparison, i.e. baseline (full image) vs pipeline (YOLO crop + EfficientNet)
base_acc = accuracy_score(df["true"], df["base_pred"])
pipe_acc = accuracy_score(df["true"], df["pipe_pred"])
detection_rate = df["yolo_detected"].mean()

print(f"Detection rate:       {detection_rate:.1%}")
print(f"Baseline accuracy:    {base_acc:.4f}")
print(f"Pipeline accuracy:    {pipe_acc:.4f}")
print(f"Difference:           {pipe_acc - base_acc:+.4f}")

print("\n--- Baseline ---")
print(classification_report(df["true"], df["base_pred"], zero_division=0))

print("\n--- Pipeline ---")
print(classification_report(df["true"], df["pipe_pred"], zero_division=0))